In [9]:
import sys
sys.path.insert(0, '/home/ravi/Thesis/ML ex/titanic-ml')

import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# ── Step 1: Load ─────────────────────────────────────────────
df = pd.read_csv('train.csv')
print(f"✅ Loaded: {df.shape[0]} rows, {df.shape[1]} cols")

# ── Step 2: Feature engineering ──────────────────────────────
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
df['Title']      = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['Title']      = df['Title'].replace(
    ['Lady','Countess','Capt','Col','Don','Dr',
     'Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
df['Title']      = df['Title'].replace(
    {'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs'})

# ── Step 3: Define X and y ───────────────────────────────────
features = ['Age','Fare','FamilySize','IsAlone',
            'Pclass','Sex','Embarked','Title']
X = df[features]
y = df['Survived']
print(f"✅ X shape: {X.shape}")
print(f"✅ Survival rate: {y.mean():.2f}")

# ── Step 4: Split ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✅ Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

# ── Step 5: Preprocessor ─────────────────────────────────────
def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler())
        ]), ['Age', 'Fare', 'FamilySize']),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), ['Sex', 'Embarked', 'Title'])
    ])

# ── Step 6: Logistic Regression ──────────────────────────────
print("\n--- Logistic Regression ---")
lr_pipeline = Pipeline([
    ('preprocessor', make_preprocessor()),
    ('model', LogisticRegression(max_iter=1000))
])
lr_scores = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring='accuracy')
lr_pipeline.fit(X_train, y_train)
print(f"CV   accuracy: {lr_scores.mean():.3f} ± {lr_scores.std():.3f}")
print(f"Test accuracy: {lr_pipeline.score(X_test, y_test):.3f}")

# ── Step 7: Random Forest ────────────────────────────────────
print("\n--- Random Forest ---")
rf_pipeline = Pipeline([
    ('preprocessor', make_preprocessor()),
    ('model', RandomForestClassifier(random_state=42, max_features='sqrt'))
])
param_grid = {
    'model__n_estimators':      [50, 100],
    'model__max_depth':         [3, 4, 5],
    'model__min_samples_split': [5, 10],
    'model__min_samples_leaf':  [2, 4]
}
grid_search = GridSearchCV(rf_pipeline, param_grid, cv=5,
                           scoring='accuracy', n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)
print(f"CV   accuracy: {grid_search.best_score_:.3f}")
print(f"Test accuracy: {grid_search.best_estimator_.score(X_test, y_test):.3f}")
print(f"Best params:   {grid_search.best_params_}")

# ── Step 8: Save ─────────────────────────────────────────────
with open('best_model.pkl', 'wb') as f:
    pickle.dump(grid_search.best_estimator_, f)
print("\n✅ Model saved!")

✅ Loaded: 891 rows, 12 cols
✅ X shape: (891, 8)
✅ Survival rate: 0.38
✅ Train: 712 rows | Test: 179 rows

--- Logistic Regression ---
CV   accuracy: 0.823 ± 0.024
Test accuracy: 0.844

--- Random Forest ---
CV   accuracy: 0.819
Test accuracy: 0.804
Best params:   {'model__max_depth': 5, 'model__min_samples_leaf': 4, 'model__min_samples_split': 5, 'model__n_estimators': 100}

✅ Model saved!
